In [ ]:
# Imports and setup
import os
os.chdir("/home/ubuntu/rhardy-us-east-1/code/coloncrafter")

import matplotlib.pyplot as plt
import numpy as np
import torch

from joblib import Parallel, delayed
from src.coloncrafter import ColonCrafterInference
from src.data.c3vd import C3VDDataset
from tqdm.auto import tqdm

In [ ]:
# Load the dataset
root_dir = "/home/ubuntu/rhardy-us-east-1/code/coloncrafter/example_data/c3vd"
section = "cecum_t1_a"
resize = (512, 512)
depth_scale = 655.35

dataset = C3VDDataset(
    root_dir=root_dir, 
    section=section,
    resize=resize,
    depth_scale=depth_scale,
)
print(f"Number of frames: {len(dataset)}")

def load_image(idx):
    return dataset[idx]["image"]

images = Parallel(n_jobs=-1)(
    delayed(load_image)(i) for i in tqdm(range(len(dataset)))
)
video = torch.stack(images, dim=0)

In [ ]:
# Plot a sample frame
idx = np.random.choice(len(dataset))
sample = dataset[idx]

fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].imshow(sample["image"].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
axs[1].imshow(sample["depth"].cpu().numpy())
axs[0].set_axis_off()
axs[1].set_axis_off()
plt.show()

In [ ]:
# Load the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ColonCrafterInference.from_pretrained(
    "romainhardy/coloncrafter",
    device=device
)

In [ ]:
# Run inference
num_inference_steps = 1
window_size = 16
overlap = 8
guidance_scale = 1.0
seed = 42

video = video * 0.5 + 0.5 # Normalize to [0, 1]
video = video.to(device, dtype=torch.float16)
num_frames, _, h, w = video.shape

pred_depth, pred_disparity = model.predict_depth(
    video,
    num_inference_steps=num_inference_steps,
    window_size=window_size,
    overlap=overlap,
    guidance_scale=guidance_scale,
    seed=seed
)

In [ ]:
# Plot predictions
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(video[0].permute(1, 2, 0).cpu().numpy().astype(np.float32))
axs[1].imshow(pred_disparity[0])
axs[2].imshow(pred_depth[0])
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[2].set_axis_off()
plt.tight_layout()
plt.show()